# Stage 2: Data Quality and Cleaning

This notebook is the **Transform** stage of the Flight Reliability Intelligence ETL process.

**Input**
- The 12 monthly BTS CSV files in `data/raw/bts/2025/`

The combined yearly file `data/processed/flights_2025.csv` is **not** used here. That file is large, so this notebook reconstructs the 2025 dataset in memory by loading each monthly source file separately and then combining them.

**Final output**
- 12 processed monthly files in `data/processed/`, named `2025_01_clean.csv` through `2025_12_clean.csv`

We will inspect, validate, and understand the combined 2025 flight dataset before making any data-quality corrections.

## Working rule

Every potential issue follows this workflow:

Inspect → Identify → Understand → Decide → Transform → Validate

We will not automatically clean, fill, convert, drop, or otherwise modify data when an issue is found. Unusual values may be legitimate because of flight-data business logic. Each change requires an explicit decision.

## Planned notebook sections

1. Load the Monthly Files and Combine Them
2. Initial Dataset Overview
3. Data Type Validation
4. Missing Values Analysis
5. Duplicate Analysis
6. Categorical Column Analysis
7. Numeric Column Analysis
8. Validate the Date Range
9. Validate Monthly Coverage
10. Flight Status Validation
11. Cancellation Logic Validation
12. Diverted Flight Logic Validation
13. Airport and Route Validation
14. Split the Clean Dataset by Month
15. Export the Processed Monthly Files
16. Data Quality & Cleaning Summary

## Current scope

This notebook implements the approved Transform steps: load and combine the monthly files, inspect data types, convert clock times and `FL_DATE`, then split and export 12 processed monthly files.

## 1. Load the Monthly Files and Combine Them

We rebuild the full 2025 dataset from the original monthly source files.

The steps are:

1. Find the 12 monthly CSV files.
2. Load each monthly file into its own DataFrame.
3. Confirm that all 12 files loaded successfully.
4. Combine them into one yearly DataFrame named `flights_2025`.

No cleaning or transformation is done during this step. The data is kept as pandas reads it from the source files.

In [1]:
from pathlib import Path

import pandas as pd


We find the project root whether the notebook is run from the project root or from the `notebooks/` folder. Then we point to the monthly source folder using a project-relative path.

In [2]:
def get_project_root():
    """Return the project root folder.

    The notebook may be run from the project root or from notebooks/.
    """
    current_folder = Path.cwd()

    if current_folder.name == "notebooks":
        return current_folder.parent

    return current_folder


PROJECT_ROOT = get_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "bts" / "2025"

print("Project root:")
print(PROJECT_ROOT)
print()
print("Raw data folder:")
print(RAW_DATA_DIR.relative_to(PROJECT_ROOT))
print()
print("Folder exists:", RAW_DATA_DIR.exists())


Project root:
/home/gur/Documents/Protfolio/Flight Reliability Intelligence System/Flight-Reliability-Intelligence

Raw data folder:
data/raw/bts/2025

Folder exists: True


We look for files named `2025_01.csv` through `2025_12.csv`. This cell only lists the files. It does not read their contents yet.

In [3]:
EXPECTED_FILE_NAMES = [f"2025_{month:02d}.csv" for month in range(1, 13)]

csv_files = sorted(RAW_DATA_DIR.glob("2025_*.csv"))
found_file_names = [file_path.name for file_path in csv_files]

print("Monthly files found:")
for file_path in csv_files:
    print(f"- {file_path.name}")

print()
print("Number of files found:", len(csv_files))


Monthly files found:
- 2025_01.csv
- 2025_02.csv
- 2025_03.csv
- 2025_04.csv
- 2025_05.csv
- 2025_06.csv
- 2025_07.csv
- 2025_08.csv
- 2025_09.csv
- 2025_10.csv
- 2025_11.csv
- 2025_12.csv

Number of files found: 12


Before loading, we confirm that all 12 expected monthly files are present. If a file is missing, we stop here rather than combining an incomplete year.

In [4]:
missing_files = [file_name for file_name in EXPECTED_FILE_NAMES if file_name not in found_file_names]

if missing_files:
    print("Missing files:")
    for file_name in missing_files:
        print(f"- {file_name}")
    raise FileNotFoundError(
        "One or more expected monthly CSV files are missing. "
        "Please add the missing files before continuing."
    )

if len(csv_files) != 12:
    raise ValueError(
        f"Expected 12 monthly files, but found {len(csv_files)}."
    )

print("Confirmation: 12 monthly files were found.")


Confirmation: 12 monthly files were found.


Now we load each monthly CSV into its own DataFrame and keep those DataFrames in a list.

Each file is read separately. We do not convert types, fill missing values, or drop rows.

In [5]:
monthly_frames = []
monthly_row_counts = {}

for file_path in csv_files:
    print(f"Loading {file_path.name}...")
    month_df = pd.read_csv(file_path, low_memory=False)
    monthly_frames.append(month_df)
    monthly_row_counts[file_path.name] = len(month_df)
    print(f"  Rows: {len(month_df):,}")
    print(f"  Columns: {len(month_df.columns)}")

print()
print("Monthly DataFrames loaded:")
for file_name, row_count in monthly_row_counts.items():
    print(f"- {file_name}: {row_count:,} rows")

print()
print("Number of monthly DataFrames loaded:", len(monthly_frames))


Loading 2025_01.csv...
  Rows: 539,747
  Columns: 62
Loading 2025_02.csv...
  Rows: 504,884
  Columns: 62
Loading 2025_03.csv...
  Rows: 600,872
  Columns: 62
Loading 2025_04.csv...
  Rows: 583,950
  Columns: 62
Loading 2025_05.csv...
  Rows: 605,648
  Columns: 62
Loading 2025_06.csv...
  Rows: 611,575
  Columns: 62
Loading 2025_07.csv...
  Rows: 631,428
  Columns: 62
Loading 2025_08.csv...
  Rows: 602,378
  Columns: 62
Loading 2025_09.csv...
  Rows: 562,439
  Columns: 62
Loading 2025_10.csv...
  Rows: 605,844
  Columns: 62
Loading 2025_11.csv...
  Rows: 570,550
  Columns: 62
Loading 2025_12.csv...
  Rows: 582,304
  Columns: 62

Monthly DataFrames loaded:
- 2025_01.csv: 539,747 rows
- 2025_02.csv: 504,884 rows
- 2025_03.csv: 600,872 rows
- 2025_04.csv: 583,950 rows
- 2025_05.csv: 605,648 rows
- 2025_06.csv: 611,575 rows
- 2025_07.csv: 631,428 rows
- 2025_08.csv: 602,378 rows
- 2025_09.csv: 562,439 rows
- 2025_10.csv: 605,844 rows
- 2025_11.csv: 570,550 rows
- 2025_12.csv: 582,304 rows


We confirm that every monthly file was loaded before combining. Analysis will be performed only on the full yearly DataFrame.

In [6]:
if len(monthly_frames) != 12:
    raise ValueError(
        f"Expected 12 monthly DataFrames, but loaded {len(monthly_frames)}."
    )

print("Confirmation: all 12 monthly DataFrames were loaded successfully.")


Confirmation: all 12 monthly DataFrames were loaded successfully.


Now that all monthly files are in memory, we stack them into one yearly DataFrame named `flights_2025`.

This is a vertical combination only. No columns are added, removed, or changed.

In [7]:
flights_2025 = pd.concat(monthly_frames, ignore_index=True)

# The monthly DataFrames are no longer needed after the yearly DataFrame is created.
del monthly_frames

print("Yearly DataFrame created.")
print("Combined yearly DataFrame shape (rows, columns):", flights_2025.shape)
print("Rows:", f"{len(flights_2025):,}")
print("Columns:", len(flights_2025.columns))


Yearly DataFrame created.
Combined yearly DataFrame shape (rows, columns): (7001619, 62)
Rows: 7,001,619
Columns: 62


We display the first few records of the combined yearly DataFrame to confirm that concatenation worked.

In [ ]:
print("First 5 records of the combined yearly DataFrame:")
display(flights_2025.head())


## 2. Initial Dataset Overview

Before detailed quality checks, we look at the overall structure of the combined yearly dataset: its size, a sample of rows, and the column list with pandas-assigned data types.

This is an inspection step only. No data is changed.

In [13]:
print("Dataset shape (rows, columns):")
print(flights_2025.shape)


Dataset shape (rows, columns):
(7001619, 62)


`head()` shows the first rows so we can see how the columns look together. This is useful before we inspect each data type in a later section.

In [17]:
print("First 5 rows:")
display(flights_2025.head())
pd.set_option('display.max_columns', None)

First 5 rows:


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_NM,ORIGIN_WAC,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_NM,DEST_WAC,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_TIME_BLK,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS,DIV_REACHED_DEST,DIV_ACTUAL_ELAPSED_TIME,DIV_ARR_DELAY,DIV_DISTANCE,DIV1_AIRPORT,DIV1_AIRPORT_ID
0,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N101NN,164,14771,1477104,32457,SFO,"San Francisco, CA",CA,California,91,12478,1247805,31703,JFK,"New York, NY",NY,New York,22,1030,1024.0,-6.0,0.0,0.0,1000-1059,11.0,1035.0,1826.0,6.0,1912,1832.0,-40.0,0.0,0.0,1900-1959,0.0,NaN,0.0,342.0,308.0,291.0,2586.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N101NN,76,12478,1247805,31703,JFK,"New York, NY",NY,New York,22,14771,1477104,32457,SFO,"San Francisco, CA",CA,California,91,600,557.0,-3.0,0.0,0.0,0600-0659,16.0,613.0,919.0,6.0,940,925.0,-15.0,0.0,0.0,0900-0959,0.0,NaN,0.0,400.0,388.0,366.0,2586.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N102UW,3244,14683,1468305,33214,SAT,"San Antonio, TX",TX,Texas,74,11057,1105703,31057,CLT,"Charlotte, NC",NC,North Carolina,36,819,817.0,-2.0,0.0,0.0,0800-0859,13.0,830.0,1131.0,11.0,1206,1142.0,-24.0,0.0,0.0,1200-1259,0.0,NaN,0.0,167.0,145.0,121.0,1095.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N103NN,185,12478,1247805,31703,JFK,"New York, NY",NY,New York,22,12892,1289208,32575,LAX,"Los Angeles, CA",CA,California,91,2100,2052.0,-8.0,0.0,0.0,2100-2159,21.0,2113.0,15.0,12.0,29,27.0,-2.0,0.0,0.0,0001-0559,0.0,NaN,0.0,389.0,395.0,362.0,2475.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N103NN,2455,10721,1072102,30721,BOS,"Boston, MA",MA,Massachusetts,13,12892,1289208,32575,LAX,"Los Angeles, CA",CA,California,91,801,756.0,-5.0,0.0,0.0,0800-0859,15.0,811.0,1055.0,16.0,1140,1111.0,-29.0,0.0,0.0,1100-1159,0.0,NaN,0.0,399.0,375.0,344.0,2611.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


`info()` summarizes every column: non-null count and the data type pandas assigned when the CSV was read.

We will review whether those data types are appropriate in Section 3. For now, we only observe them.

In [15]:
flights_2025.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7001619 entries, 0 to 7001618
Data columns (total 62 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   YEAR                     int64  
 1   QUARTER                  int64  
 2   MONTH                    int64  
 3   DAY_OF_MONTH             int64  
 4   DAY_OF_WEEK              int64  
 5   FL_DATE                  object 
 6   OP_UNIQUE_CARRIER        object 
 7   OP_CARRIER_AIRLINE_ID    int64  
 8   OP_CARRIER               object 
 9   TAIL_NUM                 object 
 10  OP_CARRIER_FL_NUM        int64  
 11  ORIGIN_AIRPORT_ID        int64  
 12  ORIGIN_AIRPORT_SEQ_ID    int64  
 13  ORIGIN_CITY_MARKET_ID    int64  
 14  ORIGIN                   object 
 15  ORIGIN_CITY_NAME         object 
 16  ORIGIN_STATE_ABR         object 
 17  ORIGIN_STATE_NM          object 
 18  ORIGIN_WAC               int64  
 19  DEST_AIRPORT_ID          int64  
 20  DEST_AIRPORT_SEQ_ID      int64  
 21  DEST_CIT

## 3. Data Type Inspection and Conversion

Pandas assigned a type to every column when the CSV files were read. Those assigned types are not always the best match for the business meaning of the field.

This section follows:

Inspect → Identify → Propose Conversion → Wait for Approval → Convert → Validate

We inspect first. We do not convert columns automatically. After the findings are reviewed, approved conversions can be applied one field at a time.

We start with a compact list of every column and the data type pandas currently assigned. This is the baseline before any conversion.

In [ ]:
print("Current data type of every column:")
print()
print(flights_2025.dtypes)


### 3.1 Date field: `FL_DATE`

`FL_DATE` is the flight date. It currently appears as `object`, which usually means pandas stored it as text.

A date field is more useful as a pandas datetime type, but only if the values can be parsed safely.

This cell shows the current type, a small sample, and how many values are missing. No conversion is applied yet.

In [ ]:
print("Column:", "FL_DATE")
print("Current data type:", flights_2025["FL_DATE"].dtype)
print("Missing values:", f"{flights_2025['FL_DATE'].isna().sum():,}")
print()
print("Sample values:")
print(flights_2025["FL_DATE"].head(10).tolist())


We now test whether the text values can be parsed as dates.

The sample values look like `1/1/2025 12:00:00 AM`. That is a date plus a midnight timestamp, which is typical for BTS flight-date fields. The time portion appears to be a placeholder, not an actual departure time.

`errors="coerce"` turns unreadable values into `NaT` so we can count them. The result is stored in a temporary Series named `fl_date_parsed`. It is **not** written back into `flights_2025`.

In [ ]:
# Temporary parsed copy for inspection only. The original FL_DATE column is not changed.
fl_date_parsed = pd.to_datetime(
    flights_2025["FL_DATE"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce",
)

invalid_dates = flights_2025.loc[fl_date_parsed.isna(), "FL_DATE"]

print("Values that could not be parsed as dates:", f"{fl_date_parsed.isna().sum():,}")
print()

if len(invalid_dates) > 0:
    print("Sample of values that could not be converted:")
    print(invalid_dates.head(10).tolist())
else:
    print("All FL_DATE values could be parsed with the expected format.")

print()
print("If converted, the date range would be:")
print("Minimum:", fl_date_parsed.min())
print("Maximum:", fl_date_parsed.max())
print("Unique dates:", fl_date_parsed.nunique())


**Finding for `FL_DATE`**

- Current type: `object` (text).
- The values look like dates with a midnight clock time, for example `1/1/2025 12:00:00 AM`.
- That midnight time is not a departure or arrival time. The actual clock times are in other columns.

**Proposed conversion (not applied yet)**

- Convert `FL_DATE` to pandas datetime, then keep only the date (`datetime64[ns]` or `.dt.date`).
- Use the format `%m/%d/%Y %I:%M:%S %p` so parsing is explicit and consistent.

No conversion is applied until this proposal is approved.

### 3.2 Clock-time fields (HHMM)

These fields represent a time of day, not a duration and not a calendar date:

- `CRS_DEP_TIME` — scheduled departure time
- `DEP_TIME` — actual departure time
- `WHEELS_ON` — wheels-on time
- `ARR_TIME` — actual arrival time
- `CRS_ARR_TIME` — scheduled arrival time

BTS often stores these as `HHMM` numbers, for example `530` for 05:30 and `1435` for 14:35.

A generic `pd.to_datetime()` conversion is not appropriate here, because `530` is not a datetime string.

First we confirm the exact column names in the dataset.

In [ ]:
clock_time_columns = [
    "CRS_DEP_TIME",
    "DEP_TIME",
    "WHEELS_ON",
    "ARR_TIME",
    "CRS_ARR_TIME",
]

print("Requested clock-time columns found in the dataset:")
for column_name in clock_time_columns:
    print(f"- {column_name}: {'found' if column_name in flights_2025.columns else 'NOT FOUND'}")

print()
print("Related column also present:")
print("- WHEELS_OFF:", "WHEELS_OFF" in flights_2025.columns)


For each clock-time column we inspect:

- current data type
- a few representative values
- missing-value count
- minimum and maximum
- whether `2400` exists (BTS sometimes uses `2400` for midnight at the end of the operating day)
- whether any value has minutes greater than 59, is negative, or is greater than 2400

These checks only describe the current values. They do not change the DataFrame.

In [ ]:
for column_name in clock_time_columns:
    column = flights_2025[column_name]

    print("=" * 60)
    print("Column:", column_name)
    print("Current data type:", column.dtype)
    print("Missing values:", f"{column.isna().sum():,}")
    print("Sample values:", column.head(8).tolist())

    non_missing = column.dropna()
    print("Minimum:", non_missing.min())
    print("Maximum:", non_missing.max())
    print("Count of 2400 values:", f"{(column == 2400).sum():,}")
    print("Count of values > 2400:", f"{(non_missing > 2400).sum():,}")
    print("Count of values < 0:", f"{(non_missing < 0).sum():,}")
    print("Count of minutes > 59:", f"{((non_missing % 100) > 59).sum():,}")
    print()


**Finding for clock-time fields**

These columns are stored as numbers (`int64` or `float64`), not as clock times.

Typical meaning of the number:

- `530` → 05:30
- `1435` → 14:35
- `1` → 00:01
- `2400` → 24:00, which BTS uses for midnight at the end of the operating day

`float64` appears on actual-operation fields such as `DEP_TIME` because cancelled or diverted flights can be missing. Pandas uses `NaN` for those missing values, so the whole column becomes float.

**Why a generic datetime conversion is not safe**

- `pd.to_datetime(530)` does not mean 05:30.
- `2400` is not a valid `HH:MM` time in pandas (`24:00` does not exist).
- Missing values on cancelled flights should remain missing, not become `00:00`.

**Proposed conversion (not applied yet)**

If conversion is approved, the simplest readable approach is:

1. Keep missing values as missing.
2. Treat `2400` as `00:00`.
3. Convert remaining `HHMM` numbers by splitting hours (`value // 100`) and minutes (`value % 100`).
4. Store the result as a pandas time or as a `HH:MM` text time, not as a full datetime.

`WHEELS_OFF` follows the same HHMM pattern and can be included if you want all clock-time fields treated the same way.

No conversion is applied until this proposal is approved.

### 3.3 Time block fields

`DEP_TIME_BLK` and `ARR_TIME_BLK` may represent scheduled time intervals, not a single clock time.

A value such as `0600-0659` means "between 06:00 and 06:59". That is a category, not a timestamp.

We inspect the current type and the unique values before deciding whether to keep them as text.

In [ ]:
time_block_columns = ["DEP_TIME_BLK", "ARR_TIME_BLK"]

for column_name in time_block_columns:
    column = flights_2025[column_name]

    print("=" * 60)
    print("Column:", column_name)
    print("Current data type:", column.dtype)
    print("Missing values:", f"{column.isna().sum():,}")
    print("Number of unique values:", column.nunique(dropna=False))
    print()
    print("Unique values:")
    print(sorted(column.dropna().unique().tolist()))
    print()


**Finding for time block fields**

- Current type: `object` (text).
- The values are intervals such as `0001-0559` and `0600-0659`.
- They are not individual clock times and should not be converted with the HHMM clock-time logic.

**Recommendation (not applied yet)**

Keep `DEP_TIME_BLK` and `ARR_TIME_BLK` as categorical/text fields.

Extracting start and end times is possible later, but it is not required for a first analytical model. The block values already support grouping by time of day.

No transformation is applied until this recommendation is approved.

### 3.4 Duration field: `CRS_ELAPSED_TIME`

`CRS_ELAPSED_TIME` is the scheduled elapsed time of the flight. In BTS On-Time Performance data, this is a **duration in minutes**, not a clock time.

It should stay conceptually separate from fields such as `CRS_DEP_TIME` and `CRS_ARR_TIME`.

This cell inspects the current type and value range only.

In [ ]:
column = flights_2025["CRS_ELAPSED_TIME"]

print("Column: CRS_ELAPSED_TIME")
print("Current data type:", column.dtype)
print("Missing values:", f"{column.isna().sum():,}")
print()
print("Sample values:")
print(column.head(10).tolist())
print()
print("Minimum:", column.min())
print("Maximum:", column.max())
print("Mean:", round(column.mean(), 2))
print("Median:", column.median())


**Finding for `CRS_ELAPSED_TIME`**

- Current type: `float64`.
- The values look like durations in minutes (for example `167` means 2 hours and 47 minutes), not `HHMM` clock times.
- `float64` is understandable if any values are missing. If every non-missing value is a whole number, `Int64` would also be reasonable.

**Proposed type (not applied yet)**

Keep `CRS_ELAPSED_TIME` as a numeric duration in minutes. Do not convert it to a clock time or a datetime.

No conversion is applied until this proposal is approved.

### 3.5 Review of remaining columns

The remaining columns are grouped by business meaning. A numeric pandas type does not always mean the field is a measurement. IDs that contain only digits are still identifiers.

In [ ]:
print("Remaining columns and current pandas types:")
print()

already_reviewed = [
    "FL_DATE",
    "CRS_DEP_TIME",
    "DEP_TIME",
    "WHEELS_ON",
    "ARR_TIME",
    "CRS_ARR_TIME",
    "DEP_TIME_BLK",
    "ARR_TIME_BLK",
    "CRS_ELAPSED_TIME",
]

remaining_columns = [column for column in flights_2025.columns if column not in already_reviewed]

for column_name in remaining_columns:
    print(f"{column_name:26s} {str(flights_2025[column_name].dtype)}")


**Conceptual classification**

This is a meaning-based grouping, not a conversion.

**Calendar attributes** — already numeric and look appropriate as integers  
`YEAR`, `QUARTER`, `MONTH`, `DAY_OF_MONTH`, `DAY_OF_WEEK`

**Airline / flight identifiers** — numeric-looking values that should not be treated as measurements  
`OP_CARRIER_AIRLINE_ID`, `OP_CARRIER_FL_NUM`, `ORIGIN_AIRPORT_ID`, `ORIGIN_AIRPORT_SEQ_ID`, `ORIGIN_CITY_MARKET_ID`, `ORIGIN_WAC`, `DEST_AIRPORT_ID`, `DEST_AIRPORT_SEQ_ID`, `DEST_CITY_MARKET_ID`, `DEST_WAC`, `DIV1_AIRPORT_ID`

**Categorical / text**  
`OP_UNIQUE_CARRIER`, `OP_CARRIER`, `TAIL_NUM`, `ORIGIN`, `ORIGIN_CITY_NAME`, `ORIGIN_STATE_ABR`, `ORIGIN_STATE_NM`, `DEST`, `DEST_CITY_NAME`, `DEST_STATE_ABR`, `DEST_STATE_NM`, `CANCELLATION_CODE`, `DIV1_AIRPORT`

**Clock time (same HHMM pattern as Section 3.2)**  
`WHEELS_OFF`

**Duration in minutes**  
`TAXI_OUT`, `TAXI_IN`, `DEP_DELAY`, `DEP_DELAY_NEW`, `ARR_DELAY`, `ARR_DELAY_NEW`, `ACTUAL_ELAPSED_TIME`, `AIR_TIME`, `CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY`, `DIV_ACTUAL_ELAPSED_TIME`, `DIV_ARR_DELAY`

**Numeric measurements**  
`DISTANCE`, `DIV_DISTANCE`, `DIV_AIRPORT_LANDINGS`

**Binary flags** — currently `float64` because missing values exist  
`CANCELLED`, `DIVERTED`, `DEP_DEL15`, `ARR_DEL15`, `DIV_REACHED_DEST`

**Proposed conversions for later approval**

- `FL_DATE`: convert to datetime/date.
- HHMM clock times: convert with a custom hours/minutes split, including `2400` handling.
- Time blocks: keep as text/category.
- `CRS_ELAPSED_TIME` and other duration fields: keep as numeric minutes.
- ID columns: keep as identifiers, not as numeric measures.
- Binary flags: optionally convert `0.0`/`1.0` to integer flags later, without filling missing values.

No remaining column is converted in this section.

### 3.6 Approved conversion: clock-time fields

The inspection showed that these fields are BTS `HHMM` local clock times, not pandas datetime values.

Approved columns to convert:

- `CRS_DEP_TIME` — scheduled departure time
- `DEP_TIME` — actual departure time
- `WHEELS_OFF` — actual takeoff time
- `WHEELS_ON` — actual landing time
- `CRS_ARR_TIME` — scheduled arrival time
- `ARR_TIME` — actual arrival time

Conversion rules:

- `530` becomes `05:30`
- `1435` becomes `14:35`
- `2400` becomes `00:00` (BTS midnight at the end of the operating day)
- Missing values stay missing
- Values that cannot represent a valid clock time are shown and the conversion stops

`DEP_TIME_BLK` and `ARR_TIME_BLK` stay as text. They are not converted.

Before changing any column, we check for invalid `HHMM` values so the conversion cannot silently create extra missing values.

We keep a snapshot of the original clock-time values and missing-value counts, and of the time-block unique values. That lets us validate the conversion without guessing what the columns looked like before.

In [ ]:
clock_time_columns = [
    "CRS_DEP_TIME",
    "DEP_TIME",
    "WHEELS_OFF",
    "WHEELS_ON",
    "CRS_ARR_TIME",
    "ARR_TIME",
]

clock_time_purpose = {
    "CRS_DEP_TIME": "Scheduled departure time",
    "DEP_TIME": "Actual departure time",
    "WHEELS_OFF": "Actual takeoff time",
    "WHEELS_ON": "Actual landing time",
    "CRS_ARR_TIME": "Scheduled arrival time",
    "ARR_TIME": "Actual arrival time",
}

# Snapshot used later to confirm time blocks were not changed.
dep_time_blk_before = sorted(flights_2025["DEP_TIME_BLK"].dropna().unique().tolist())
arr_time_blk_before = sorted(flights_2025["ARR_TIME_BLK"].dropna().unique().tolist())

original_clock_sample = flights_2025[clock_time_columns].head(10).copy()
original_missing_counts = {
    column_name: flights_2025[column_name].isna().sum()
    for column_name in clock_time_columns
}

print("Original clock-time sample:")
display(original_clock_sample)

print()
print("Original missing-value counts:")
for column_name, missing_count in original_missing_counts.items():
    print(f"- {column_name}: {missing_count:,}")


This helper converts one BTS `HHMM` number to a `HH:MM` text time.

It is intentionally small so it can later be copied into the Python ETL script.

Missing values are returned as missing. `2400` is treated as `00:00`. Any other value that cannot be a valid clock time is marked as failed, not silently turned into midnight.

In [ ]:
def convert_bts_hhmm(hhmm_value):
    """Convert one BTS HHMM value to a HH:MM text time.

    Examples:
        530  -> 05:30
        1435 -> 14:35
        2400 -> 00:00
    """
    if pd.isna(hhmm_value):
        return pd.NA

    hhmm_value = int(hhmm_value)

    # BTS uses 2400 for midnight at the end of the operating day.
    if hhmm_value == 2400:
        return "00:00"

    hours = hhmm_value // 100
    minutes = hhmm_value % 100

    if hhmm_value < 0 or hours > 23 or minutes > 59:
        return "INVALID"

    return f"{hours:02d}:{minutes:02d}"


We first look for values that cannot be converted. If any exist, we display them and stop. We do not convert the columns until every non-missing value is a valid `HHMM` time or the approved special case `2400`.

In [ ]:
invalid_hhmm_found = False

for column_name in clock_time_columns:
    converted_preview = flights_2025[column_name].map(convert_bts_hhmm)
    invalid_rows = flights_2025.loc[converted_preview == "INVALID", [column_name]]

    print(f"{column_name}: invalid HHMM values = {len(invalid_rows):,}")

    if len(invalid_rows) > 0:
        invalid_hhmm_found = True
        print("Sample of values that cannot be converted:")
        display(invalid_rows.head(10))

if invalid_hhmm_found:
    raise ValueError(
        "Some HHMM values cannot be converted to a valid clock time. "
        "No clock-time columns were changed. Please review the invalid values before continuing."
    )

print()
print("All non-missing clock-time values are valid HHMM times or the special value 2400.")
print("It is safe to convert these columns.")


Now we convert the six approved clock-time columns in place. Missing values stay missing. Time-block columns are not included in this loop.

In [ ]:
for column_name in clock_time_columns:
    print(f"Converting {column_name}...")
    flights_2025[column_name] = flights_2025[column_name].map(convert_bts_hhmm)

print()
print("Clock-time conversion complete.")
print("Time-block columns were not converted.")


We validate each converted column: purpose, new data type, a small sample, missing values after conversion, and whether any extra missing values were created.

If conversion created more missing values than before, we stop and inspect those rows.

In [ ]:
extra_missing_found = False

for column_name in clock_time_columns:
    converted_column = flights_2025[column_name]
    original_missing = original_missing_counts[column_name]
    new_missing = converted_column.isna().sum()
    extra_missing = new_missing - original_missing

    print("=" * 60)
    print("Column:", column_name)
    print("Purpose:", clock_time_purpose[column_name])
    print("New data type:", converted_column.dtype)
    print("Missing values after conversion:", f"{new_missing:,}")
    print("Original missing values:", f"{original_missing:,}")
    print("Additional missing values created by conversion:", f"{extra_missing:,}")
    print()
    print("Sample of converted values:")
    print(converted_column.head(10).tolist())
    print()

    if extra_missing != 0:
        extra_missing_found = True
        print("Conversion changed the missing-value count. Please inspect this column before continuing.")
        print()

    failed_values = converted_column[converted_column == "INVALID"]
    if len(failed_values) > 0:
        extra_missing_found = True
        print("Values marked INVALID after conversion:")
        print(failed_values.head(10).tolist())
        print()

if extra_missing_found:
    raise ValueError(
        "Clock-time conversion created unexpected missing or invalid values. "
        "Please inspect the output above before deciding how to handle them."
    )

print("Validation passed: no extra missing values were created.")


We also show the original sample next to the converted sample so the `HHMM` to `HH:MM` mapping is easy to check.

In [ ]:
print("Original sample:")
display(original_clock_sample)

print()
print("Converted sample:")
display(flights_2025[clock_time_columns].head(10))


Finally, we confirm that the time-block fields were left in their original text format, such as `0600-0659`.

In [ ]:
print("DEP_TIME_BLK data type:", flights_2025["DEP_TIME_BLK"].dtype)
print("ARR_TIME_BLK data type:", flights_2025["ARR_TIME_BLK"].dtype)
print()

dep_time_blk_after = sorted(flights_2025["DEP_TIME_BLK"].dropna().unique().tolist())
arr_time_blk_after = sorted(flights_2025["ARR_TIME_BLK"].dropna().unique().tolist())

print("DEP_TIME_BLK unique values unchanged:", dep_time_blk_before == dep_time_blk_after)
print("ARR_TIME_BLK unique values unchanged:", arr_time_blk_before == arr_time_blk_after)
print()
print("DEP_TIME_BLK unique values:")
print(dep_time_blk_after)
print()
print("ARR_TIME_BLK unique values:")
print(arr_time_blk_after)


### 3.7 Approved conversion: `FL_DATE`

`FL_DATE` is still stored as text, for example `1/1/2025 12:00:00 AM`. The midnight clock time is a placeholder, not a departure time.

The approved conversion is:

- Parse `FL_DATE` as a pandas datetime using the format `%m/%d/%Y %I:%M:%S %p`
- Keep it as a date/datetime type
- Preserve missing values
- Stop and display any values that cannot be parsed, instead of silently turning them into nulls

We save the original `FL_DATE` sample and missing-value count so we can validate the conversion afterward.

In [ ]:
original_fl_date_sample = flights_2025["FL_DATE"].head(10).copy()
original_fl_date_dtype = flights_2025["FL_DATE"].dtype
original_fl_date_missing = flights_2025["FL_DATE"].isna().sum()

print("Column: FL_DATE")
print("Purpose: Flight date")
print("Current data type:", original_fl_date_dtype)
print("Original missing values:", f"{original_fl_date_missing:,}")
print()
print("Original sample:")
print(original_fl_date_sample.tolist())


We parse the dates into a temporary Series first. The original column is not changed until we confirm that every non-missing value can be converted.

In [ ]:
fl_date_parsed = pd.to_datetime(
    flights_2025["FL_DATE"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce",
)

# Values that had text but could not be parsed.
failed_dates = flights_2025.loc[
    flights_2025["FL_DATE"].notna() & fl_date_parsed.isna(),
    "FL_DATE",
]

print("Values that could not be parsed as dates:", f"{len(failed_dates):,}")

if len(failed_dates) > 0:
    print()
    print("Sample of values that cannot be converted:")
    print(failed_dates.head(10).tolist())
    raise ValueError(
        "Some FL_DATE values cannot be parsed. "
        "The column was not changed. Please review the invalid values before continuing."
    )

print("All non-missing FL_DATE values can be parsed with the expected format.")
print("It is safe to convert this column.")


Now we write the parsed dates back to `FL_DATE`. The source values include `12:00:00 AM`, so the converted datetime values will be at midnight. That midnight time is still only a date placeholder, not a flight clock time.

In [ ]:
flights_2025["FL_DATE"] = fl_date_parsed

print("FL_DATE conversion complete.")
print("New data type:", flights_2025["FL_DATE"].dtype)


We validate the conversion: new data type, date range, missing values, and whether any extra nulls were created.

In [ ]:
converted_fl_date = flights_2025["FL_DATE"]
new_missing = converted_fl_date.isna().sum()
extra_missing = new_missing - original_fl_date_missing

print("Column: FL_DATE")
print("Purpose: Flight date")
print("New data type:", converted_fl_date.dtype)
print()
print("Original sample:")
print(original_fl_date_sample.tolist())
print()
print("Converted sample:")
print(converted_fl_date.head(10).tolist())
print()
print("Minimum date:", converted_fl_date.min())
print("Maximum date:", converted_fl_date.max())
print("Unique dates:", converted_fl_date.nunique())
print()
print("Original missing values:", f"{original_fl_date_missing:,}")
print("Missing values after conversion:", f"{new_missing:,}")
print("Additional missing values created by conversion:", f"{extra_missing:,}")

if extra_missing != 0:
    raise ValueError(
        "FL_DATE conversion created unexpected missing values. "
        "Please inspect the output above before deciding how to handle them."
    )

print()
print("Validation passed: FL_DATE was converted without creating extra missing values.")


## 4. Split the Final Dataset by Month

The approved transformations are complete. We do not export one large yearly file.

Instead, we split `flights_2025` back into 12 monthly DataFrames using the month in `FL_DATE`. Each record should belong to exactly one month, and no rows should be lost.

Before splitting, we confirm that `FL_DATE` is a datetime column. The split uses `.dt.month`, so the conversion from Section 3 must already be applied.

In [ ]:
print("Yearly DataFrame shape:", flights_2025.shape)
print("FL_DATE data type:", flights_2025["FL_DATE"].dtype)
print("FL_DATE missing values:", f"{flights_2025['FL_DATE'].isna().sum():,}")

if "datetime" not in str(flights_2025["FL_DATE"].dtype):
    raise TypeError(
        "FL_DATE must be a datetime column before splitting by month. "
        "Please run the approved FL_DATE conversion first."
    )


We create 12 monthly DataFrames, one for each month of 2025. This is only a split. No additional cleaning is applied.

In [ ]:
yearly_row_count = len(flights_2025)
monthly_clean = {}

for month in range(1, 13):
    month_df = flights_2025[flights_2025["FL_DATE"].dt.month == month].copy()
    monthly_clean[month] = month_df
    print(f"Month {month:02d}: {len(month_df):,} rows")


We validate the split before exporting:

- all 12 months are present
- each record belongs to exactly one month
- the monthly row counts add up to the yearly row count
- no records were left unassigned

In [ ]:
monthly_row_counts = {month: len(month_df) for month, month_df in monthly_clean.items()}
total_monthly_rows = sum(monthly_row_counts.values())
months_present = [month for month, row_count in monthly_row_counts.items() if row_count > 0]
unassigned_rows = flights_2025[flights_2025["FL_DATE"].isna() | ~flights_2025["FL_DATE"].dt.month.between(1, 12)]

print("Months present:", months_present)
print("Number of months present:", len(months_present))
print("Yearly rows:", f"{yearly_row_count:,}")
print("Sum of monthly rows:", f"{total_monthly_rows:,}")
print("Unassigned rows:", f"{len(unassigned_rows):,}")
print()
print("Rows by month:")
for month in range(1, 13):
    print(f"- {month:02d}: {monthly_row_counts[month]:,}")

if len(months_present) != 12:
    raise ValueError("The split does not contain all 12 months.")

if len(unassigned_rows) != 0:
    raise ValueError("Some records were not assigned to a month. The split is incomplete.")

if total_monthly_rows != yearly_row_count:
    raise ValueError(
        "The monthly row counts do not match the yearly DataFrame. "
        "Records were lost or duplicated during the split."
    )

print()
print("Split validation passed. No records were lost.")


## 5. Export the Processed Monthly Files

We export each monthly DataFrame to `data/processed/` as `2025_MM_clean.csv`.

The export writes the data as it currently exists after the approved conversions. The pandas index is not included.

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

exported_files = []
total_rows_exported = 0

for month in range(1, 13):
    file_name = f"2025_{month:02d}_clean.csv"
    output_file = processed_dir / file_name
    month_df = monthly_clean[month]

    month_df.to_csv(output_file, index=False)

    exported_files.append(file_name)
    total_rows_exported += len(month_df)
    print(f"Exported {file_name}: {len(month_df):,} rows")

print()
print("Export complete.")
print("Number of files exported:", len(exported_files))
print("Total rows exported:", f"{total_rows_exported:,}")
print("Output directory:", processed_dir.relative_to(PROJECT_ROOT))


# Data Quality & Cleaning Summary

This section is an audit trail of the Transform stage. It includes only work that was actually inspected, approved, and applied.

## Dataset dimensions

- Original combined 2025 dataset: 7,001,619 rows and 62 columns
- Final processed dataset: the same 7,001,619 rows and 62 columns
- No rows were removed during this stage

## Data type issues discovered

- `FL_DATE` was stored as text (`object`), for example `1/1/2025 12:00:00 AM`
- Clock-time fields were stored as BTS `HHMM` numbers, not as clock times
- Actual-operation clock times used `float64` because cancelled or diverted flights can be missing
- `DEP_TIME_BLK` and `ARR_TIME_BLK` are text intervals such as `0600-0659`, not individual clock times
- `CRS_ELAPSED_TIME` is a duration in minutes, not a clock time

## Data type conversions actually performed

- `FL_DATE` converted to `datetime64[ns]`
- `CRS_DEP_TIME`, `DEP_TIME`, `WHEELS_OFF`, `WHEELS_ON`, `CRS_ARR_TIME`, and `ARR_TIME` converted from `HHMM` to `HH:MM` text
- `2400` stored as `00:00`
- Missing clock times kept as missing

## Values intentionally left unchanged

- `DEP_TIME_BLK` and `ARR_TIME_BLK` remain text time blocks
- `CRS_ELAPSED_TIME` remains a numeric duration in minutes
- Identifier columns remain identifiers, not numeric measures
- Missing values were not filled or dropped
- Duplicates, outliers, cancellation logic, diversion logic, and airport/route checks were not treated in this stage

## Date-range and monthly coverage used for export

- Converted `FL_DATE` values range from 2025-01-01 to 2025-12-31
- The processed yearly DataFrame was split back into 12 monthly files using `FL_DATE`

## Final output files

The processed files are saved in `data/processed/`:

- `2025_01_clean.csv` through `2025_12_clean.csv`

The same approved ETL logic is also available as the executable script `python/etl_flights.py`.